# NAME:SOWMYA HARDAGERI
# SRN: PES2UG23CS590

Unit 2 - Part 3a: Chain of Thought (CoT)

In [1]:
%pip install python-dotenv --upgrade --quiet langchain langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.5/500.5 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.1/158.1 kB 12.7 MB/s eta 0:00:00


In [2]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_groq import ChatGroq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

# Using Llama3.1-8b (Small/Fast) to demonstrate logic failures
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)

Enter your Groq API Key: ··········


In [3]:
question = "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many does he have now?"

Standard Prompt for Roger Question

In [4]:
# 1. Standard Prompt (Direct Answer)
prompt_standard = f"Answer this question: {question}"
print("--- STANDARD (Llama3.1-8b) ---")
print(llm.invoke(prompt_standard).content)

--- STANDARD (Llama3.1-8b) ---
To find out how many tennis balls Roger has now, we need to add the initial number of tennis balls he had (5) to the number of tennis balls he bought (2 cans * 3 tennis balls per can).

2 cans * 3 tennis balls per can = 6 tennis balls

Now, let's add the initial number of tennis balls (5) to the number of tennis balls he bought (6):

5 + 6 = 11

So, Roger now has 11 tennis balls.


Chain of Thought Prompt for Roger Question

In [5]:
# 2. CoT Prompt (Magic Phrase)
prompt_cot = f"Answer this question. Let's think step by step. {question}"

print("--- Chain of Thought (Llama3.1-8b) ---")
print(llm.invoke(prompt_cot).content)

--- Chain of Thought (Llama3.1-8b) ---
To find out how many tennis balls Roger has now, we need to follow these steps:

1. Roger already has 5 tennis balls.
2. He buys 2 more cans of tennis balls. Each can has 3 tennis balls, so he buys 2 x 3 = 6 more tennis balls.
3. Now, we add the tennis balls he already had (5) to the tennis balls he bought (6). 5 + 6 = 11

So, Roger now has 11 tennis balls.


Tree of Thought Prompt for Roger Question

In [10]:
#3. ToT Prompt
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# Keep temperature slightly higher for branch diversity
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)

question = "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many does he have now?"

# ----------------------------
# Step 1: Generate 3 reasoning branches
# ----------------------------

prompt_branch = ChatPromptTemplate.from_template(
    "Solve the problem using a different reasoning approach.\nProblem: {question}\nApproach {id}:"
)

branches = RunnableParallel(
    approach1=prompt_branch.partial(id="1") | llm | StrOutputParser(),
    approach2=prompt_branch.partial(id="2") | llm | StrOutputParser(),
    approach3=prompt_branch.partial(id="3") | llm | StrOutputParser(),
)

# ----------------------------
# Step 2: Judge selects best reasoning
# ----------------------------

prompt_judge = ChatPromptTemplate.from_template(
    """
    We have three different solutions to this problem:

    Problem: {question}

    1: {approach1}
    2: {approach2}
    3: {approach3}

    Act as a Math Teacher.
    - Identify which reasoning is the clearest.
    - Confirm the correct final answer.
    """
)

tot_chain = (
    RunnableParallel(question=RunnableLambda(lambda x: x), branches=branches)
    | (lambda x: {**x["branches"], "question": x["question"]})
    | prompt_judge
    | llm
    | StrOutputParser()
)

print("\n--- ToT (Math Problem) ---\n")
print(tot_chain.invoke(question))


--- ToT (Math Problem) ---

Class! Today we're going to analyze three different approaches to solving a simple problem: Roger has 5 tennis balls and buys 2 more cans of 3 tennis balls each. Let's break down each approach and see which one is the clearest.

**Approach 1: Step-by-Step Approach**

This approach is straightforward and easy to follow. It breaks down the problem into manageable steps:

1. Roger initially has 5 tennis balls.
2. He buys 2 more cans of tennis balls. Each can has 3 tennis balls, so he buys 2 x 3 = 6 tennis balls.
3. To find the total number of tennis balls Roger has now, we add the number of tennis balls he initially had (5) to the number of tennis balls he bought (6). total_tennis_balls = 5 + 6 = 11

This approach is clear and easy to understand, but let's see if we can simplify it further.

**Approach 2: Breaking Down the Problem into Smaller Steps**

This approach is similar to the first one, but it breaks down the problem in a slightly different way:

1. Ro

Graph of Thought Prompt for Roger Question

In [11]:
#4. GoT Prompt
# Use moderate creativity
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)

# ----------------------------
# Step 1: Generate multiple reasoning styles
# ----------------------------

prompt_styles = ChatPromptTemplate.from_template(
    "Solve this problem using {style} reasoning:\n{question}"
)

drafts = RunnableParallel(
    algebra=prompt_styles.partial(style="algebraic") | llm | StrOutputParser(),
    visual=prompt_styles.partial(style="visual explanation") | llm | StrOutputParser(),
    logical=prompt_styles.partial(style="step-by-step logical") | llm | StrOutputParser(),
)

# ----------------------------
# Step 2: Combine into unified explanation
# ----------------------------

prompt_combine = ChatPromptTemplate.from_template(
    """
    We have three reasoning styles for solving:

    {question}

    Algebraic: {algebra}
    Visual: {visual}
    Logical: {logical}

    Combine the strongest parts of each explanation into one final clear solution.
    Provide the final answer.
    """
)

got_chain = (
    RunnableParallel(question=RunnableLambda(lambda x: x), drafts=drafts)
    | (lambda x: {**x["drafts"], "question": x["question"]})
    | prompt_combine
    | llm
    | StrOutputParser()
)

print("\n--- GoT (Math Problem) ---\n")
print(got_chain.invoke(question))



--- GoT (Math Problem) ---

To solve the problem of how many tennis balls Roger has now, let's combine the strongest parts of the algebraic, visual, and logical explanations.

**Step 1: Identify the initial number of tennis balls**
Roger initially has 5 tennis balls.

**Step 2: Calculate the number of tennis balls in the new cans**
He buys 2 more cans of tennis balls, each can having 3 tennis balls. To find the total number of tennis balls in the cans, we multiply the number of cans by the number of tennis balls per can: 2 cans * 3 tennis balls/can = 6 tennis balls.

**Step 3: Add the initial number of tennis balls to the number of tennis balls in the new cans**
Now, let's add the initial number of tennis balls (5) to the number of tennis balls in the new cans (6): 5 (initial tennis balls) + 6 (new tennis balls) = 11.

Therefore, Roger now has **11 tennis balls**.


Summary Table for Roger Question

| Feature                 | CoT                        | ToT                                    | GoT                                |
| ----------------------- | -------------------------- | -------------------------------------- | ---------------------------------- |
| Prompt Type             | "Let's think step by step" | Branch + Judge                         | Branch + Merge                     |
| Reasoning Structure     | Linear chain               | Tree (multiple paths)                  | Graph (parallel + combine)         |
| Number of Thought Paths | 1                          | 3                                      | 3                                  |
| Evaluation Step         |  No                       |  Yes (Teacher selects best)           |  No selection (Synthesis instead) |
| Output Length           | Moderate                   | Long (meta-analysis included)          | Concise but refined                |
| Cognitive Pattern       | Sequential reasoning       | Divergent → Evaluative                 | Divergent → Convergent             |
| Best For                | Logical problems           | Decision-making / comparing strategies | Integrating multiple perspectives  |
| Creativity Level        | Low (temp=0)               | Moderate (temp=0.7)                    | Moderate (temp=0.7)                |
| Final Answer            | 11                         | 11                                     | 11                                 |


Unit 2 - Part 3b: Tree of Thoughts (ToT) & Graph of Thoughts (GoT)

In [6]:
%pip install python-dotenv --upgrade --quiet langchain langchain-groq

In [7]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_groq import ChatGroq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

# Using Llama3.1-8b
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)

Tree of Thought Prompt for Vegetables question

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

problem = "How can I get my 5-year-old to eat vegetables?"

# Step 1: The Branch Generator
prompt_branch = ChatPromptTemplate.from_template(
    "Problem: {problem}. Give me one unique, creative solution. Solution {id}:"
)

branches = RunnableParallel(
    sol1=prompt_branch.partial(id="1") | llm | StrOutputParser(),
    sol2=prompt_branch.partial(id="2") | llm | StrOutputParser(),
    sol3=prompt_branch.partial(id="3") | llm | StrOutputParser(),
)

# Step 2: The Judge
prompt_judge = ChatPromptTemplate.from_template(
    """
    I have three proposed solutions for: '{problem}'

    1: {sol1}
    2: {sol2}
    3: {sol3}

    Act as a Child Psychologist. Pick the most sustainable one (not bribery) and explain why.
    """
)

# Chain: Input -> Branches -> Judge -> Output
tot_chain = (
    RunnableParallel(problem=RunnableLambda(lambda x: x), branches=branches)
    | (lambda x: {**x["branches"], "problem": x["problem"]})
    | prompt_judge
    | llm
    | StrOutputParser()
)

print("--- Tree of Thoughts (ToT) Result ---")
print(tot_chain.invoke(problem))

--- Tree of Thoughts (ToT) Result ---
After analyzing the three proposed solutions, I would recommend Solution 3: Create a 'Vegetable Garden' in a Cup as the most sustainable approach to encourage a 5-year-old to eat vegetables. Here's why:

1. **Long-term impact**: This solution has a long-term impact on a child's eating habits and relationship with food. By involving your child in the process of growing their own vegetables, you're teaching them about the journey from seed to plate, which can foster a deeper appreciation for healthy food.
2. **Ownership and responsibility**: By giving your child ownership of the plant, you're encouraging them to take responsibility for its growth and care. This sense of ownership can translate to a greater willingness to try and eat the vegetables they've grown.
3. **Learning and exploration**: The 'garden journal' aspect of this solution allows your child to learn about the process of growing food, track progress, and draw pictures. This hands-on ap

Chain of Thought Prompt for Vegetables question

In [12]:
from langchain_groq import ChatGroq

# Deterministic reasoning
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)

problem = "How can I get my 5-year-old to eat vegetables?"

prompt_cot = f"""
You are a child behavior expert.

Solve the following problem using clear step-by-step reasoning.
Think step by step before giving the final advice.

Problem: {problem}
"""

print("\n--- CoT (Vegetables Problem) ---\n")
print(llm.invoke(prompt_cot).content)



--- CoT (Vegetables Problem) ---

As a child behavior expert, I'd be happy to help you with this common challenge. Here's a step-by-step approach to encourage your 5-year-old to eat vegetables:

**Step 1: Understand the reasons behind their resistance**

Before we start, it's essential to understand why your child might be resistant to eating vegetables. Some common reasons include:

- Lack of exposure to new foods
- Texture or taste preferences
- Association with unpleasant experiences (e.g., being forced to eat something they didn't like)
- Limited knowledge about the benefits of vegetables

**Step 2: Make it fun and engaging**

Children are naturally curious and love to explore new things. Here are some ideas to make eating vegetables more enjoyable:

- Create a "veggie face" on their plate using sauces or dips
- Use fun shapes and colors to make vegetables more appealing (e.g., cutting carrots into sticks or making a fruit kebab)
- Play a "taste test" game where they try different

Graph of Thought Prompt for Vegetables question

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# Moderate creativity
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)

problem = "How can I get my 5-year-old to eat vegetables?"

# ----------------------------
# Step 1: Generate different strategic approaches
# ----------------------------

prompt_styles = ChatPromptTemplate.from_template(
    "Suggest a {style} strategy to solve this problem:\n{problem}"
)

drafts = RunnableParallel(
    psychological=prompt_styles.partial(style="psychological") | llm | StrOutputParser(),
    gamification=prompt_styles.partial(style="gamification") | llm | StrOutputParser(),
    environmental=prompt_styles.partial(style="environmental") | llm | StrOutputParser(),
)

# ----------------------------
# Step 2: Combine best ideas
# ----------------------------

prompt_combine = ChatPromptTemplate.from_template(
    """
    We have three strategies to solve:

    {problem}

    Psychological: {psychological}
    Gamification: {gamification}
    Environmental: {environmental}

    Combine the strongest parts of each into one practical and sustainable parenting strategy.
    """
)

got_chain = (
    RunnableParallel(problem=RunnableLambda(lambda x: x), drafts=drafts)
    | (lambda x: {**x["drafts"], "problem": x["problem"]})
    | prompt_combine
    | llm
    | StrOutputParser()
)

print("\n--- GoT (Vegetables Problem) ---\n")
print(got_chain.invoke(problem))



--- GoT (Vegetables Problem) ---

Here's a comprehensive and practical parenting strategy that combines the strongest parts of each approach:

**"Veggie Heroes" Program**

**Objective:** Encourage your 5-year-old child to develop a healthy relationship with vegetables by making mealtime enjoyable, engaging, and interactive.

**Program Structure:**

1. **Create a Positive Environment:**
   - Display a colorful vegetable arrangement on the dinner table or a nearby shelf.
   - Make the dining area visually appealing with bright colors, flowers, or plants.
   - Play soothing background music to create a positive atmosphere.

2. **Lead by Example:**
   - Eat vegetables yourself in front of your child, making it a normal part of your meals.
   - Share your favorite vegetable dishes with your child and explain why they're good for you.

3. **Involve Your Child in Food Preparation:**
   - Let your child help with meal planning, grocery shopping, and vegetable prep.
   - Make it a fun, interac

Summary Table for vegetables question
| Aspect              | CoT                                | ToT                                                 | GoT                                                   |
| ------------------- | ---------------------------------- | --------------------------------------------------- | ----------------------------------------------------- |
| Problem Used        | Vegetables (5-year-old)            | Same problem                                        | Same problem                                          |
| Reasoning Structure | Linear step-by-step parenting plan | 3 independent solutions → psychologist selects best | 3 strategy types → merged into one structured program |
| Output Style        | Behavioral reasoning steps         | Comparative + evaluative                            | Structured unified action plan                        |
| Meta-Evaluation     |  No                               |  Yes (chooses most sustainable)                    |  No judging, instead synthesis                       |
| Creativity Level    | Moderate                           | Higher (branch diversity)                           | Highest (integrated “Veggie Heroes” program)          |
| Organization        | Numbered steps                     | Approach 1/2/3 + evaluation                         | Program structure + objectives + tips                 |
| Decision-Making     | Single path advice                 | Multi-path + best selection                         | Multi-path + combined strengths                       |
| Practicality        | Good                               | Very thoughtful                                     | Most practical and structured                         |
| Length              | Medium                             | Long                                                | Longest and most structured                           |


Graph of Thought Prompt for Movie Plot Question

In [9]:
# 1. The Generator (Divergence)
prompt_draft = ChatPromptTemplate.from_template(
    "Write a 1-sentence movie plot about: {topic}. Genre: {genre}."
)

drafts = RunnableParallel(
    draft_scifi=prompt_draft.partial(genre="Sci-Fi") | llm | StrOutputParser(),
    draft_romance=prompt_draft.partial(genre="Romance") | llm | StrOutputParser(),
    draft_horror=prompt_draft.partial(genre="Horror") | llm | StrOutputParser(),
)

# 2. The Aggregator (Convergence)
prompt_combine = ChatPromptTemplate.from_template(
    """
    I have three movie ideas for the topic '{topic}':
    1. Sci-Fi: {draft_scifi}
    2. Romance: {draft_romance}
    3. Horror: {draft_horror}

    Your task: Create a new Mega-Movie that combines the TECHNOLOGY of Sci-Fi, the PASSION of Romance, and the FEAR of Horror.
    Write one paragraph.
    """
)

# 3. The Chain
got_chain = (
    RunnableParallel(topic=RunnableLambda(lambda x: x), drafts=drafts)
    | (lambda x: {**x["drafts"], "topic": x["topic"]})
    | prompt_combine
    | llm
    | StrOutputParser()
)

print("--- Graph of Thoughts (GoT) Result ---")
print(got_chain.invoke("Time Travel"))

--- Graph of Thoughts (GoT) Result ---
In "Time's Echo," a brilliant physicist, Emma, discovers a hidden time machine that allows her to travel back to pivotal moments in her own past. As she navigates through different eras, she finds herself falling for a charming and younger version of her older self, who is on the verge of a tragic accident that will erase her from existence forever. However, their budding romance is disrupted by a malevolent entity that begins to stalk Emma across different timelines, manipulating events to ensure her demise. With each iteration, Emma is forced to relive the same day, trying to outsmart the entity and prevent her own destruction, all while navigating the complexities of her own love life and the fragile fabric of time.


Chain of Thought Prompt for Movie Plot Question

In [14]:
from langchain_groq import ChatGroq

# Deterministic creativity
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)

topic = "Time Travel"

prompt_cot = f"""
You are a professional screenwriter.

Create a movie plot about {topic}.
Think step by step:

1. Define the protagonist.
2. Define the central conflict.
3. Introduce a twist.
4. Describe the climax.
5. End with an emotional hook.

Write the final plot in one paragraph.
"""

print("\n--- CoT (Movie Plot) ---\n")
print(llm.invoke(prompt_cot).content)



--- CoT (Movie Plot) ---

Here's the movie plot for a time travel story:

Meet Dr. Emma Taylor, a brilliant and determined physicist who has spent her entire career studying the mysteries of time travel. After years of tireless work, Emma finally succeeds in building a functioning time machine, but her excitement is short-lived as she soon discovers that her invention has been stolen by a rival scientist, Dr. Marcus Thompson, who plans to use it to alter historical events for his own gain. As Emma frantically tries to track down her stolen invention, she realizes that Marcus has already traveled back in time to the 1960s, where he is manipulating key events to reshape the course of history. But here's the twist: Emma soon discovers that she has a personal connection to Marcus, who is actually her long-lost brother, thought to have died in a tragic accident when they were children. As Emma navigates the complex web of time travel and family secrets, she must confront her own dark past 

Tree of Thought Prompt for Movie plot question

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# Higher temperature for creative diversity
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.8)

topic = "Time Travel"

# ----------------------------
# Step 1: Generate 3 movie concepts
# ----------------------------

prompt_branch = ChatPromptTemplate.from_template(
    "Create a unique movie plot about {topic}. Concept {id}:"
)

branches = RunnableParallel(
    concept1=prompt_branch.partial(id="1") | llm | StrOutputParser(),
    concept2=prompt_branch.partial(id="2") | llm | StrOutputParser(),
    concept3=prompt_branch.partial(id="3") | llm | StrOutputParser(),
)

# ----------------------------
# Step 2: Film Critic Judge
# ----------------------------

prompt_judge = ChatPromptTemplate.from_template(
    """
    I have three movie concepts about {topic}:

    1: {concept1}
    2: {concept2}
    3: {concept3}

    Act as a professional film critic.
    - Choose the most compelling concept.
    - Explain why it would succeed critically and commercially.
    """
)

tot_chain = (
    RunnableParallel(topic=RunnableLambda(lambda x: x), branches=branches)
    | (lambda x: {**x["branches"], "topic": x["topic"]})
    | prompt_judge
    | llm
    | StrOutputParser()
)

print("\n--- ToT (Movie Plot) ---\n")
print(tot_chain.invoke(topic))



--- ToT (Movie Plot) ---

After analyzing the three concepts, I firmly believe that **Concept 2: "Echoes of Eternity"** has the most compelling narrative, character development, and thematic resonance. Here's why:

**Compelling Concept:**

"Echoes of Eternity" explores the intricacies of time travel, identity, and consciousness in a thought-provoking and visually stunning way. The concept revolves around Dr. Maya Singh, a brilliant physicist who discovers a way to send her consciousness back in time using the Chrono-Resonator. However, her actions create a duplicate consciousness, stuck in a temporal loop, forcing her to confront the consequences of her choices.

**Character Development:**

Maya's character is richly layered, and her journey is both emotionally resonant and intellectually stimulating. Her struggles with her duplicate consciousnesses, her desire to correct the mistakes of the past, and her growing understanding of the complexities of time and identity make her a relata

Summary Table for Movie Question (Time Travel)
| Aspect              | CoT                      | ToT                                  | GoT                                 |
| ------------------- | ------------------------ | ------------------------------------ | ----------------------------------- |
| Problem Used        | Time Travel movie        | Same topic                           | Same topic                          |
| Reasoning Structure | Linear story development | 3 concepts → critic selects best     | 3 genre drafts → merged into one    |
| Output Style        | Structured narrative arc | Analytical + evaluative review       | Synthesized cinematic story         |
| Creativity Level    | Moderate                 | High (multiple concepts)             | Very High (cross-genre fusion)      |
| Meta-Evaluation     |  No                     |  Yes (critic justifies choice)      |  No selection, instead combination |
| Complexity          | One timeline conflict    | Deep thematic + commercial analysis  | Multi-genre layered narrative       |
| Business Insight    | Minimal                  | Included Oscar + box office analysis | Focused on story only               |
| Output Length       | Medium                   | Longest                              | Medium-long                         |
| Refinement Level    | Clean single story       | Highly analytical                    | Most imaginative                    |


#Final Summary and Comparison Table
Chain-of-Thought (CoT) improves reasoning by breaking problems into sequential steps.
Tree-of-Thought (ToT) enhances decision-making by generating multiple solution branches and selecting the best one.
Graph-of-Thought (GoT) enables idea synthesis by merging multiple independent thought paths into a cohesive output.
Compared to standard prompting, advanced prompting techniques significantly improve reasoning depth, creativity, and structured outputs.

| Feature          | Standard Prompt | CoT                 | ToT               | GoT                    |
| ---------------- | --------------- | ------------------- | ----------------- | ---------------------- |
| Reasoning Style  | Direct answer   | Linear step-by-step | Branch + Evaluate | Branch + Combine       |
| Structure        | Single path     | Sequential          | Tree structure    | Graph structure        |
| Creativity       | Low             | Moderate            | High              | Very High              |
| Evaluation Step  | No              | No                  | Yes               | No (Synthesis instead) |
| Best For         | Simple Q&A      | Logical problems    | Decision-making   | Creative synthesis     |
| Temperature Used | 0.0             | 0.0                 | 0.7               | 0.7                    |
